# Lecture 8 — Reference Genome Indexing, STAR Alignment & Post-Alignment QC

Align sequencing reads to a reference genome with **STAR**, on free Google Colab.

**Data** — GSE129221, sample `GSM3702795` ("A1 PC9 cells"): 5,000 read pairs at 150 bp. Reference: GRCh38 chr17, chr21, chr22 = 180.8 Mb (Ensembl release 115).

**The constraint is compute, not memory.** A whole-genome STAR index needs ~28 GB of RAM — but the reason it cannot run here is just as much the **hours of CPU** it takes to build on two vCPUs. Cutting the reference to three chromosomes shrinks index time, index size and peak RAM *together*:

| | 3 chromosomes | whole genome |
|---|---|---|
| index build (16 threads) | **77 s** | hours |
| index on disk | **1.7 GB** | ~28 GB |
| peak RAM | **4.1 GB** | ~28 GB |

**A second limit, and it is not memory either** — our reads come from the whole transcriptome, but only 3 of 24 chromosomes are indexed. Roughly **90% of reads cannot map**, and every unmappable read costs a full search before STAR discards it. Throughput is ~210 read pairs/second, which is why we align 5,000 pairs rather than the full 1.5 M. Expect a mapping rate near **11%**, not 90% — that is correct for a reduced reference, not a failure.

**What we produce** — a sorted BAM, alignment QC, and gene-level counts from featureCounts.


## Part A — Inputs and pre-flight

### A0 — Upload the data

Download **`lecture-08-data.tar.gz`** (56 MB):

[Google Drive — lecture-08-data.tar.gz](https://drive.google.com/file/d/1RCf4kJOXrEvZdOjLWc18GdRdHdcq6D2S/view?usp=sharing)

In Colab, open the **Files** panel in the left sidebar (`⋮` → *Upload*), or just drag the file onto the notebook. It should land at:

```
/content/lecture-08-data.tar.gz
```

Check it arrived whole before unpacking — a partial upload fails halfway through and leaves a broken `data/`:

```bash
stat -c%s /content/lecture-08-data.tar.gz     # expect 57779840
```

Then run the next cell to unpack.

In [ ]:
%%bash

cd /content
# A1 — unpack
tar -xzf lecture-08-data.tar.gz
du -sh data/*

### A2 — Install the tools

Colab ships none of the bioinformatics tools, and there is no conda or micromamba on the image. We install from Ubuntu's `apt` archive, which pins the versions for us:

| package | provides | version |
|---|---|---|
| `rna-star` | STAR | 2.7.11b |
| `samtools` | samtools | 1.19.2 |
| `subread` | featureCounts | 2.0.6 |

In [ ]:
%%bash

apt-get update -qq
apt-get install -y -qq rna-star samtools subread
STAR --version

### A3 — Pre-flight

Before spending 25 minutes on an index, prove the inputs are intact. Three checks:

1. **Mate counts match.** Both files must have the same number of records, divisible by 4.
2. **The first record is readable.** Header, sequence, `+`, quality — four lines.
3. **Contig names match.** The FASTA headers and the GTF's first column must use `17`, `21`, `22` — not `chr17`.

If any check fails, fix it now. STAR will not fix it for you.


In [ ]:
%%bash

cd /content

echo "=== A3.1 mate counts ==="
for f in data/raw_fastq/*.sub.fq; do
  n=$(wc -l < "$f")
  echo "$(basename "$f"): $((n / 4)) reads"
done

echo
echo "=== A3.2 first record ==="
head -4 data/raw_fastq/SRR8836894_R1.sub.fq

echo
echo "=== A3.3 contigs ==="
echo "FASTA:"
grep '^>' data/ref/Homo_sapiens.GRCh38.dna.primary_assembly.sub.fa
echo "GTF:"
cut -f1 data/gtf/Homo_sapiens.GRCh38.sub.gtf | grep -v '^#' | sort -u

echo
echo "The two lists must be identical."


> **⏸ STOP — return to the slides.**
>
> We read the FastQC panel next, then the alignment problem. Do not build the index yet — we start it together in Part B.


## Part B — Build the index

STAR's index is the expensive step. It builds a **suffix array** over the reference, and sorting that array is what dominates the runtime.

For our 181 Mb reference the finished index is about 1.7 GB. A whole-genome index would be ~28 GB and take hours on two vCPUs — it could not be built here at all. That is the entire reason the three-chromosome reference makes this run possible.

Three parameters catch people out.

**`--genomeSAindexNbases`** — STAR's default of 14 is sized for the full genome. On a 181 Mb reference it is too large and STAR warns it may seg-fault at the mapping step. The correct value scales with reference length (`log2(len)/2 - 1`), which is **12** here.

**`--sjdbGTFfile`** — this is the RNA-seq part, and leaving it out is the mistake that makes alignment crawl. Without annotated splice junctions STAR has to discover every intron *de novo*, searching large genomic windows for every batch of reads. With the GTF baked into the index those junctions are already known. `--sjdbOverhang` must be one less than the read length — **149** for 150 bp reads.

**`--genomeSAsparseD`** — the memory-for-speed dial. The suffix array is `(1 + 8/sparseD) x genomeLength`, so `1` costs ~1.6 GB and `2` costs ~0.9 GB. Memory is not the scarce resource here, so take the faster `1`.

Expect this to take several minutes. The index is deterministic, so if you need to re-run later, keep `data/star_index/` and skip straight to step 2.


In [ ]:
%%bash

cd /content

sparseD=1            # dense suffix array: ~1.6 GB instead of 0.9 GB, and skips the sparse de-sparsify pass. Memory is not the scarce resource, so take the faster option.
saIndexNbases=12     # STAR's recommendation for a 181 Mb genome. The default 14 is too large here and seg-faults at mapping.

rm -rf data/star_index
mkdir -p data/star_index

# Launch in the background: the class returns to the slides while this runs.
nohup STAR --runThreadN 2 \
     --runMode genomeGenerate \
     --genomeDir data/star_index \
     --genomeFastaFiles data/ref/Homo_sapiens.GRCh38.dna.primary_assembly.sub.fa \
     --sjdbGTFfile data/gtf/Homo_sapiens.GRCh38.sub.gtf \
     --sjdbGTFfeatureExon exon \
     --sjdbOverhang 149 \
     --genomeSAsparseD $sparseD \
     --genomeSAindexNbases $saIndexNbases \
     > data/star_index_build.log 2>&1 &

echo "STAR genomeGenerate started (PID $!)."
echo "Detailed log: data/star_index/Log.out"
sleep 2
tail -n 5 data/star_index/Log.out 2>/dev/null || echo "(Log.out not written yet)"
ls -lh data/star_index/


> **⏸ STOP — the build is running. Return to the slides.**
>
> The build takes ~25 minutes on 2 vCPUs. **Do not wait for it.** We spend exactly that long on the STAR algorithm, then come back to Part C.
>
> To check progress from the **Terminal** panel:
>
> ```bash
> tail -f /content/data/star_index/Log.out
> ```
>
> `Ctrl+C` stops the tail, not the build. The build is finished when `Log.out` prints `done` and `data/star_index/SA` exists.


## Part C — Align the reads

STAR loads the index into memory once, then streams the reads through it. Three settings match our production pipeline:

| setting | why |
|---|---|
| `--outSAMtype BAM SortedByCoordinate` | coordinate-sorted output — what Qualimap expects, and it saves a separate samtools sort |
| `--outFilterType BySJout` | keeps alignments consistent with the splice-junction filters; it is **not** a uniqueness filter |
| `--alignSJoverhangMin 8` / `--alignSJDBoverhangMin 1` | require at least 8 spliced bases at novel junctions, 1 at annotated ones |

Alignment memory is fixed by the index (~3.2 GB) and does **not** grow with read count — 5,000 pairs costs the same RAM as 1.5 M, only less time. At roughly 210 pairs per second this takes a minute and a half.


In [ ]:
%%bash

cd /content

dataDir=data
indexDir=$dataDir/star_index

if [ ! -f "$indexDir/SA" ]; then
  echo "Index not ready: $indexDir/SA is missing."
  echo "Wait for the Part B build to finish, then re-run this cell."
  exit 1
fi
alignDir=$dataDir/STAR
fqDir=$dataDir/raw_fastq
threads=2                         # Colab has 2 vCPU
sjOverhangMin=8                   # drop junctions anchored on fewer than 8 spliced bases
sjdbOverhangMin=1                 # more permissive: 1 spliced base at annotated junctions

r1=$(ls $fqDir | grep -E '_R1\.sub\.fq$' | sed "s|^|$fqDir/|")
r2=$(ls $fqDir | grep -E '_R2\.sub\.fq$' | sed "s|^|$fqDir/|")

echo "R1: $r1"
echo "R2: $r2"

mkdir -p $alignDir

STAR --runThreadN $threads \
     --runMode alignReads \
     --genomeDir $indexDir \
     --readFilesIn $r1 $r2 \
     --outFileNamePrefix $alignDir/sample. \
     --outSAMtype BAM SortedByCoordinate \
     --outFilterType BySJout \
     --outFilterMultimapNmax 20 \
     --alignSJoverhangMin $sjOverhangMin \
     --alignSJDBoverhangMin $sjdbOverhangMin

ls -lh $alignDir/


To watch Part C while it runs — STAR writes the BAM only at the end, so watch its progress file, not the output directory:

```bash
watch -n 2 'cat /content/data/STAR/sample.Log.progress.out'
```

`Ctrl+C` stops it.

> **⏸ STOP — return to the slides.**
>
> Leave `sample.Log.final.out` open. We read it together, then dissect the BAM.


## Part D — Inspect the BAM

The BAM is the evidence. Before trusting the summary, decode one record yourself.

- `samtools view -H` — the header, including the exact `@PG` command.
- `samtools view | head -1` — one record; find FLAG, CIGAR, MAPQ.
- `samtools view | awk '$6 ~ /N/'` — a spliced read.
- `samtools view -c` vs `-F 256` — all records vs primary.


In [ ]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

echo "=== header (first 8 lines) ==="
samtools view -H "$bam" | head -8

echo
echo "=== first record ==="
samtools view "$bam" | head -1

echo
echo "=== spliced reads (CIGAR contains N) ==="
samtools view "$bam" | awk '$6 ~ /N/' | head -3

echo
echo "=== all vs primary ==="
echo "all records:     $(samtools view -c "$bam")"
echo "primary records: $(samtools view -c -F 256 "$bam")"


> **⏸ STOP — return to the slides.**
>
> We do QC next: `Log.final.out`, the denominator arithmetic, `flagstat`, and `idxstats`. You will need this BAM.


## Part E — QC the alignment

Three sources, none of which needs an install:

- **`sample.Log.final.out`** — STAR's own summary: mapping rate, splice junctions, mismatch rates
- **`samtools flagstat`** — mapped / properly paired / duplicates
- **`samtools idxstats`** — reads per contig

`idxstats` is the one that earns its place here. Only 3 of 24 chromosomes are indexed, so roughly **90% of reads have nowhere to land**, and `--outFilterType BySJout` gives a unique-mapping rate near **11% — the expected result, not a failure**. `idxstats` shows exactly where the reads that did map went.

For a real sample you would reach for two more tools:

| tool | what it adds | note |
|---|---|---|
| **Qualimap** | RNA-seq QC: 5′/3′ bias, duplication rate, coverage uniformity, insert size | what our production pipeline uses; not in Colab's `apt` archive, so it needs conda or the official jar |
| **MultiQC** | aggregates STAR, samtools and picard metrics into one HTML report | what our pipeline finishes with |

Neither is installed here on purpose: at 5,000 read pairs these metrics are not statistically meaningful, and the install cost is real.


In [ ]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

# The .bai enables idxstats and region queries.
samtools index "$bam"

echo "=== STAR summary ==="
grep -E 'input reads|Uniquely mapped reads %|mapped to multiple loci|splices: Total|Mismatch rate' data/STAR/sample.Log.final.out

echo
echo "=== flagstat ==="
samtools flagstat "$bam"

echo
echo "=== reads per contig (idxstats) ==="
samtools idxstats "$bam"

echo
echo "=== insert size ==="
samtools stats "$bam" | grep 'insert size'


### Part E2 — Count the genes

`featureCounts` intersects each fragment with the exons in the GTF and assigns it to a `gene_id`.

- `-p --countReadPairs` — paired-end, count fragments not mates.
- `-s 0` — unstranded. If the library is stranded, `1` or `2`; the wrong value halves the counts.
- `-t exon -g gene_id` — count exons, aggregate by gene.

Read the `.summary` file, not just the matrix. `Assigned` is the numerator; everything else is a diagnosis.


In [ ]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

featureCounts -p --countReadPairs -T 2 -s 0 \
  -t exon -g gene_id \
  -a data/gtf/Homo_sapiens.GRCh38.sub.gtf \
  -o data/STAR/counts.txt \
  "$bam"

echo
echo "=== featureCounts summary ==="
cat data/STAR/counts.txt.summary


### Reproduce the arithmetic

Before you believe the BAM, make the numbers close:

```text
5,000 input fragments
  x (11.06% unique + 1.24% multi) = 615 fragments
  x 2 mates                        = 1,230 primary records
  + 242 secondary                  = 1,472 records
```

Then check the MAPQ distribution:

```bash
samtools view data/STAR/sample.Aligned.sortedByCoord.out.bam | cut -f5 | sort -n | uniq -c
```

> **⏸ STOP — done.**
>
> Homework: decode one spliced record, explain why 11% is the right mapping rate for this reference, and bring one QC number you would report in a methods section.
